# Help Assistant Chatbot — Architecture & Agentic RAG (learning notebook)

This notebook is a **parallel** guide to the codebase: how pieces fit together, and **runnable** snippets for local experiments.

**Prerequisites**

- Project created with **UV** (or `pip install -r requirements.txt` / `uv sync`).
- A `.env` in the project root with at least: `API_URL` (OpenAI-style chat URL ending in `.../chat/completions` — may include `/v1` or `/v2`), `API_KEY` if your gateway needs it, `LLM_MODEL`.
- For the **LangGraph** path, the HTTP API must support **tool / function calling** on chat completions.

Run notebooks from the **`notebooks/`** folder, or from project root; the first code cell adds the project root to `sys.path` and loads `.env`.

## 1. Mental model: static KB vs agent

- **Vector store (Chroma + BM25):** document chunks you uploaded; *static* between uploads. Hybrid search = dense + keyword, merged (RRF).
- **SQLite `cases`:** *dynamic* list/create support cases (separate from PDF index).
- **Chat with `use_rag=True`:** a **LangGraph** ReAct **agent** chooses **tools**:
  - `search_knowledge_base` — hybrid RAG over Chroma
  - `get_open_active_cases` — read `Active` rows from SQLite
  - `create_active_support_case` — insert a new `Active` case
- **Chat with `use_rag=False`:** one plain LLM call (no tools, no RAG) via `CustomLLM` in `llm/client.py`.

Read **`AGENTIC_RAG.md`**, **`RAG_TUNING.md`**, **`SETUP_DATABASE.md`** in the repo for detail.

### Flow (Mermaid — renders in VS Code / Jupyter with Mermaid support)

```mermaid
flowchart LR
  U[User query] --> A[FastAPI /chat]
  A -->|use_rag true| G[LangGraph ReAct agent]
  G --> T1[search_knowledge_base]
  G --> T2[get_open_active_cases]
  G --> T3[create_active_support_case]
  T1 --> C[(Chroma + BM25)]
  T2 --> D[(SQLite cases.db)]
  T3 --> D
  G --> L[ChatOpenAI at API_URL base]
  A -->|use_rag false| L0[CustomLLM single call]
```


## 2. Setup: project root, `sys.path`, `.env`

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=True)

print("Project root:", ROOT)
print("API_URL (truncated):", (os.environ.get("API_URL") or "")[:80], "...")

## 3. `API_URL` → OpenAI client `base_url`

`ChatOpenAI` posts to `{base_url}/chat/completions`. Your env may store the **full** `.../v2/chat/completions` URL; we strip only `/chat/completions` so `/v2` is **not** duplicated as `/v2/v1` (see `llm/openai_compat.py`).

In [ ]:
from llm.openai_compat import get_openai_compatible_base_url

raw = __import__("os").environ.get("API_URL", "")
base = get_openai_compatible_base_url()
print("Full API_URL in .env (first 100 chars):", raw[:100] if raw else "(not set)")
print("Resolved base for LangChain client:", base or "(empty — set API_URL)")

## 4. SQLite cases (no LLM) — `list_cases` & mock seed

Seeding (shell): `uv run python -m database.seed_mock_data` from project root.

In [ ]:
from database import init_db, list_cases

init_db()
active = list_cases(status="Active")
all_rows = list_cases(status=None)
print(f"Active cases: {len(active)} | All cases: {len(all_rows)}")
for row in active[:5]:
    print(row.get("id"), row.get("title", "")[:60])

## 5. One-shot RAG (eval-style, no agent)

`eval_retrieve_and_build_prompt` + `rag_helpers.retrieve_hybrid` match the **static** retrieve → prompt used in evaluation; not the same as the ReAct tool loop, but the **retrieval** math is the same as inside `search_knowledge_base`.

In [ ]:
from services.rag import eval_retrieve_and_build_prompt
from services.rag_helpers import retrieve_hybrid

q = "What is IMF?"  # change to match your documents
results, context, _ = retrieve_hybrid(q, top_k=3)
print("Number of hits:", len(results))
print("Context preview (500 chars):\n", (context or "")[:500])

## 6. Full chat — LangGraph agent (`use_rag=True` default)

Requires a working **tool-calling** chat API. This runs `run_support_agent` → `chat_response` behavior.

In [ ]:
import asyncio
from services.rag import chat_response

query = "List all open active cases."
text, citations = asyncio.run(chat_response(query, use_rag=True, num_results=5, temperature=0.3))
print(text)
print("Citations (from last KB search, may be empty):", len(citations), "items")

## 7. Chat with RAG off — single LLM (`use_rag=False`)

No LangGraph; useful when your gateway does not support `tools`.

In [ ]:
import asyncio
from services.rag import chat_response

text, _ = asyncio.run(chat_response(
    "Say hello in one sentence.",
    use_rag=False,
    temperature=0.5,
))
print(text)

## 8. (Optional) HTTP `requests` to a running local API

Start the server: `uv run uvicorn api.main:app --host 0.0.0.0 --port 8000` from project root. Then set `API_BASE` below if different.

In [ ]:
import json
import os
import requests

API_BASE = os.environ.get("API_BASE", "http://127.0.0.1:8000")
try:
    h = requests.get(f"{API_BASE}/health", timeout=2)
    print("Health:", h.status_code, h.text[:200])
    r = requests.get(f"{API_BASE}/cases", params={"status": "Active"}, timeout=5)
    print("GET /cases?status=Active:", r.status_code, len(r.json() if r.ok else []), "rows")
    body = {
        "query": "List open active cases with one-line summary if possible.",
        "use_rag": True,
        "num_results": 5,
        "temperature": 0.3,
    }
    c = requests.post(f"{API_BASE}/chat", json=body, timeout=120)
    print("POST /chat:", c.status_code)
    if c.ok:
        data = c.json()
        print(data.get("response", "")[:800])
except Exception as e:
    print("Skip or start API:", e)

## 9. File map (for navigation)

| Area | Main files |
|------|------------|
| API | `api/main.py`, `api/routes/chat.py`, `api/routes/cases.py`, `api/routes/documents.py` |
| Agent | `services/agent_graph.py`, `services/agent_tools.py` |
| RAG + cache | `services/rag.py`, `services/rag_helpers.py` |
| LLM (non-agent) | `llm/client.py` |
| OpenAI base URL | `llm/openai_compat.py` |
| Chroma + BM25 | `vector_store/store.py` |
| SQLite | `database/schema.py`, `database/cases_repo.py`, `database/seed_mock_data.py` |
| Tuning / DB docs | `RAG_TUNING.md`, `AGENTIC_RAG.md`, `SETUP_DATABASE.md` |